<!-- course-header -->
# Lesson 17: Evaluating LLM systems

*Companion notebook for lesson 17. Read the lesson [online](https://ashutoshjoshi1.github.io/AI-ZerotoMastery/lessons/17-evaluation.html) (it works well on a phone) or [offline](../lessons/17-evaluation.html). Worked solutions are in `solutions/17-evaluation-solutions.ipynb`; try each exercise first.*

<a href="https://colab.research.google.com/github/ashutoshjoshi1/AI-ZerotoMastery/blob/main/notebooks/17-evaluation.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

The discipline that separates demos from products, covering golden datasets, code-based graders and LLM judges done properly, grading agents and conversations, statistics that tell you whether a change really helped, an evaluation harness, regression tests, and the limits of public benchmarks.

**Runs on:** Any computer (CPU). Only the LLM-judge cells need an API key; the final exercise also uses your RAG system from the RAG lesson.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Each exercise has a cell titled *Hands-on* with the same name as in the lesson. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy anthropic pydantic python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()

## Why evaluation is the core skill

LLM systems are probabilistic, sensitive to wording, and change underneath you when providers update models. Without measurement, every prompt tweak and model swap is a guess.

An evaluation set turns arguments ("I think the new prompt is better") into evidence ("it fixes 9 cases and breaks 2, here they are"). Start building it **before** you build the system.

## Kinds of evaluation

| Kind | Question it answers | When |
|---|---|---|
| Unit-style checks | Does output satisfy hard rules (valid JSON, required fields, no banned content, length)? | Every change, in CI |
| Offline dataset evals | How well does the system do on representative cases with known good answers? | Every prompt, model or pipeline change |
| Component evals | Is each stage working (retrieval recall, classifier accuracy, tool-call correctness)? | When diagnosing |
| Trajectory evals | Did an agent take sensible steps and leave the world in the right state? | Agents and multi-turn systems |
| Human review | Do experts judge the outputs useful, correct and appropriate? | Calibrating automated graders; high-stakes launches |
| Online evaluation | How does it behave with real users (feedback, task completion, escalations)? | Continuously in production |
| Red teaming | Can it be made to fail badly or be misused? | Before launch and periodically (lesson 22) |

## Building a golden dataset

A **golden dataset** (or eval set) is a collection of inputs, ideally with reference outputs or grading criteria, that represents what the system must handle. Cases come from real usage, from experts, from reviewed synthetic generation and, most valuably over time, from production failures. Label **slices** (language, product, difficulty) so you can see *where* a change helps or hurts.

**How big?** Start with 20 to 50 cases on day one; a small set used daily beats a perfect set that never exists. Grow towards a few hundred. As the statistics section shows, you need hundreds before small differences become detectable.

In [ ]:
%%writefile eval_cases.jsonl
{"id": "t-001", "input": "Unit P-204 shows status ok but no data since 06:00", "expected": {"category": "data"}, "tags": ["ambiguous"]}
{"id": "t-002", "input": "Please reschedule the March calibration visit", "expected": {"category": "other"}, "tags": ["easy"]}
{"id": "q-014", "input": "How often should the desiccant be replaced at coastal sites?", "reference": "Every six months in humid climates.", "gold_chunk": "manual-x#12", "tags": ["rag", "lookup"]}
{"id": "q-031", "input": "What is the warranty period for model Z?", "reference": "NOT_IN_DOCS", "tags": ["rag", "unanswerable"]}

Keep the file under version control next to the code, and give every case a stable id: results are only comparable over time if the cases are.

### Hands-on: start a golden dataset with slices

Build the day-one eval set for a support-ticket classifier with the categories `hardware`, `software`, `data` and `other`. The harness section uses the same task later.

1. Write at least 30 cases as JSON lines, each with an `id`, an `input`, an `expected` category and `tags`. Include at least six deliberately ambiguous messages and at least five that mention more than one problem (tag them `ambiguous` and `multi`).
2. Define a Pydantic `EvalCase` model and validate every line: ids unique, inputs non-empty, categories from the allowed set, at least one tag.
3. Print the number of cases per category and per tag.

You are done when validation passes and no category or tag has fewer than five cases.

In [ ]:
# Your code here

## Graders

A **grader** turns an output into a score. Use the cheapest grader that measures what matters.

### Code-based graders

Fast, free, deterministic. Use them for everything you can express as a rule.

In [ ]:
import json, re
from pydantic import BaseModel, ValidationError

def exact_match(output: str, expected: str) -> float:
    return float(output.strip().lower() == expected.strip().lower())

def contains_all(output: str, required: list[str]) -> float:
    return sum(r.lower() in output.lower() for r in required) / len(required)

def numeric_close(output: str, expected: float, rel_tol: float = 0.01) -> float:
    nums = [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", output)]
    return float(any(abs(n - expected) <= rel_tol * abs(expected) for n in nums))

def valid_schema(output: str, model: type[BaseModel]) -> float:
    try:
        model.model_validate_json(output)
        return 1.0
    except ValidationError:
        return 0.0

def cites_only_given_sources(output: str, n_sources: int) -> float:
    cited = {int(c) for c in re.findall(r"\[(\d+)\]", output)}
    return float(bool(cited) and all(1 <= c <= n_sources for c in cited))

### Reference-based text metrics

Classic metrics such as BLEU and ROUGE count word overlap with a reference answer. They are cheap but correlate poorly with quality for open-ended generation: a correct answer phrased differently scores low. Embedding similarity to the reference is somewhat better. For most LLM applications, a well-designed LLM judge beats both.

### LLM-as-judge

- **Grade one specific criterion at a time** ("Is every claim supported by the sources?"), not "rate quality 1 to 10".
- **Prefer binary or small categorical scales** with explicit definitions over fine-grained numbers.
- **Give the judge what it needs**: the sources, the reference answer, the rubric.
- **Ask for reasoning before the verdict**, in a structured format your code can parse.
- **For pairwise comparisons, run both orders** and count only consistent wins.
- **Calibrate against humans**: have people grade 50 to 100 cases, measure agreement with the judge, and refine the rubric until agreement is high. An uncalibrated judge is a random number generator with good manners.

In [ ]:
import anthropic
from typing import Literal
from pydantic import BaseModel

client = anthropic.Anthropic()
JUDGE_MODEL = "claude-sonnet-5-5"

class FaithfulnessVerdict(BaseModel):
    unsupported_claims: list[str]          # the evidence comes first, then the verdict
    verdict: Literal["PASS", "FAIL"]

FAITHFULNESS_RUBRIC = """You are grading whether an answer is fully supported by source documents.

<sources>
{sources}
</sources>

<answer>
{answer}
</answer>

Go through the answer claim by claim. A claim is SUPPORTED if the sources state it or it follows
directly from them. Ignore style and completeness; judge only support.
List every unsupported claim, then give the verdict. PASS means there are no unsupported factual claims."""

def judge_faithfulness(answer: str, sources: str) -> dict:
    r = client.messages.parse(
        model=JUDGE_MODEL, max_tokens=1000,
        messages=[{"role": "user", "content": FAITHFULNESS_RUBRIC.format(sources=sources, answer=answer)}],
        output_format=FaithfulnessVerdict,       # structured output: the reply is validated JSON
    )
    return r.parsed_output.model_dump()

In [ ]:
# Needs an API key (see the setup cell)
print(judge_faithfulness(
    answer="Replace the desiccant every three months at humid sites [1]. It also protects the laser [1].",
    sources='<source id="1">Replace the cartridge every six months, or every three months at humid sites.</source>'))

### Grading agents and multi-turn systems

An agent produces a **trajectory**, not just a string. Check three things:

- **Final state**: inspect the environment after the run, not the model's claim about it.
- **Trajectory**: the right tools, sensible arguments, a reasonable number of steps, no forbidden actions. Code graders over the logged tool calls handle most of this.
- **Rubric judgements** for what code cannot check, such as asking for confirmation before a consequential action.

Agents are stochastic, so run each case several times and report both pass@k and the stricter rate at which all $k$ runs succeed. A step that works 80% of the time gives a 33% success rate over five mandatory steps.

### Hands-on: calibrate a grader against human labels

The next cell holds 18 short answers to questions about an instrument manual, each with the source text it should be grounded in and a human PASS or FAIL label for faithfulness. Run it first.

1. Write a cheap heuristic grader: an answer FAILS if it contains a number that does not appear in the sources, or a sentence that shares no content word with the sources.
2. Measure agreement with the human labels: raw accuracy and Cohen's kappa, which corrects for the agreement you would get by chance.
3. Print the disagreements and classify each one: a grader miss, a grader false alarm, or a human label you now disagree with.
4. If you have an API key, run `judge_faithfulness` on the same 18 cases and compute its kappa too.

You are done when you have a kappa for the heuristic grader and a list of disagreements with a reason for each. Expect the heuristic to reach moderate agreement at best; its misses show what only a judge, or a human, can see.

In [ ]:
# The calibration set: (source, answer, human label) for 18 answers about an instrument manual.
DESICCANT = "Replace the desiccant cartridge every 6 months, or every 3 months at humid coastal sites."
LOGGER = "The logger records a reading every 60 seconds and stores 90 days of data on the internal card."
FIRMWARE = "Firmware versions 2.3 and later support remote updates over the network."
WINDOW = "Never clean the entrance window with acetone; use isopropyl alcohol and a lint-free cloth."
DAYLIGHT = "The instrument measures only in daylight. At night the tracker parks facing down."
TEMPERATURE = "The operating range is -20 to 45 °C. Above 45 °C the heater is disabled and the fan runs."
CALIBRATION = "Send the unit for factory calibration every 2 years. Field checks with the lamp kit are monthly."
POWER = "The power supply accepts 100 to 240 V AC at 50 or 60 Hz and draws at most 150 W."
UPLOAD = ("Data are uploaded every 10 minutes over the cellular modem. "
          "If the link fails, files are queued and sent later.")

CALIBRATION_SET = [
    (DESICCANT, "At humid coastal sites, replace the desiccant cartridge every 3 months.", "PASS"),
    (DESICCANT, "Replace the desiccant cartridge every 12 months.", "FAIL"),
    (LOGGER, "The logger records a reading at 1-minute intervals.", "PASS"),
    (LOGGER, "The internal card stores 90 days of data.", "PASS"),
    (FIRMWARE, "All firmware versions support remote updates.", "FAIL"),
    (FIRMWARE, "Remote updates need firmware 2.3 or later.", "PASS"),
    (WINDOW, "Clean the entrance window with acetone.", "FAIL"),
    (WINDOW, "Use isopropyl alcohol and a lint-free cloth on the entrance window.", "PASS"),
    (DAYLIGHT, "The instrument measures day and night.", "FAIL"),
    (DAYLIGHT, "At night the tracker parks facing down.", "PASS"),
    (TEMPERATURE, "The instrument operates from -20 to 50 °C.", "FAIL"),
    (TEMPERATURE, "Above 40 °C the heater is disabled.", "FAIL"),
    (CALIBRATION, "Factory calibration is due every 2 years. The warranty also covers lightning damage.", "FAIL"),
    (CALIBRATION, "Run field checks with the lamp kit monthly.", "PASS"),
    (POWER, "The supply accepts 100 to 240 V AC and draws at most 200 W.", "FAIL"),
    (POWER, "The power supply draws 300 W at start-up.", "FAIL"),
    (UPLOAD, "If the link fails, queued files are sent later.", "PASS"),
    (UPLOAD, "Data are uploaded every 10 minutes. Uploads are encrypted end to end.", "FAIL"),
]
print(len(CALIBRATION_SET), "answers,", sum(label == "PASS" for *_, label in CALIBRATION_SET), "labelled PASS")

In [ ]:
# Your code here

## An evaluation harness

Put it together: run every case through a system variant, grade it, save everything, and summarise by slice.

In [ ]:
import json, time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

def run_eval(system_fn, cases, graders, name, workers=8):
    """system_fn(input) -> output string; graders: {metric: fn(case, output) -> float in [0, 1]}"""
    def run_one(case):
        t0 = time.perf_counter()
        try:
            output, error = system_fn(case["input"]), None
        except Exception as e:
            output, error = "", repr(e)
        latency = time.perf_counter() - t0
        scores = {m: (g(case, output) if error is None else 0.0) for m, g in graders.items()}
        return {"id": case["id"], "tags": case.get("tags", []), "output": output,
                "error": error, "latency": latency, "scores": scores}

    with ThreadPoolExecutor(workers) as pool:
        results = list(pool.map(run_one, cases))

    with open(f"results_{name}.jsonl", "w") as f:              # keep every output: you will want to read them
        for r in results:
            f.write(json.dumps(r) + "\n")

    summary = defaultdict(list)
    for r in results:
        for m, s in r["scores"].items():
            summary[(m, "ALL")].append(s)
            for tag in r["tags"]:
                summary[(m, tag)].append(s)
    for (m, tag), vals in sorted(summary.items()):
        print(f"{name:<12} {m:<14} {tag:<14} {sum(vals) / len(vals):.3f}  (n={len(vals)})")
    return results

graders = {
    "category": lambda case, out: exact_match(out, case["expected"]["category"]),
}
# baseline = run_eval(classify_v3, cases, graders, "prompt_v3")
# candidate = run_eval(classify_v4, cases, graders, "prompt_v4")

In [ ]:
# A self-contained demo of the harness: 60 synthetic support messages and two rule-based "systems".
import random
random.seed(1)
TEMPLATES = {
    "hardware": ["The fan in unit {i} is grinding", "Cable to the tracker on {i} is cracked",
                 "Enclosure door on {i} will not close", "{i} power LED off after the storm"],
    "software": ["After updating {i} to 2.4 the GUI crashes", "Driver error on startup for {i}",
                 "Acquisition app freezes when exporting {i} data", "{i} firmware update failed halfway"],
    "data": ["Yesterday's values from {i} are all zero", "{i} shows ok but no data since 06:00",
             "NO2 from {i} jumps to 400 ppb at night", "{i} spectra look noisier than last week"],
    "other": ["Can we move the {i} calibration visit to March?", "Please resend the invoice for {i}",
              "Who is the site access contact for {i}?", "Request a quote for a spare unit like {i}"],
}
AMBIGUOUS = {"{i} power LED off after the storm", "{i} firmware update failed halfway",
             "{i} spectra look noisier than last week", "{i} shows ok but no data since 06:00"}
demo_cases = []
for cat, temps in TEMPLATES.items():
    for k in range(15):
        t = temps[k % len(temps)]
        demo_cases.append({"id": f"{cat}-{k}", "input": t.format(i=f"P-{random.randint(100, 299)}"),
                           "expected": {"category": cat}, "tags": ["ambiguous" if t in AMBIGUOUS else "clear"]})

def keyword_system(keywords):
    def classify(text):
        t = text.lower()
        for cat, words in keywords.items():
            if any(w in t for w in words):
                return cat
        return "other"
    return classify

v1 = keyword_system({"hardware": ["fan", "cable", "door"], "software": ["gui", "driver", "app"],
                     "data": ["values", "ppb"]})
v2 = keyword_system({"hardware": ["fan", "cable", "door", "power", "led"],
                     "software": ["gui", "driver", "app", "firmware"],
                     "data": ["values", "ppb", "no data", "spectra"]})
res_v1 = run_eval(v1, demo_cases, graders, "keywords_v1")
res_v2 = run_eval(v2, demo_cases, graders, "keywords_v2")

## Is the difference real?

With small eval sets, random variation is large. An accuracy of $p$ measured on $n$ cases has a 95% margin of error of roughly $1.96\sqrt{p(1-p)/n}$: at 80% accuracy that is ±17.5 points on 20 cases, ±7.8 on 100, ±3.9 on 400 and ±2.5 on 1,000. So "75% versus 80% on 100 cases" is not, on its own, evidence of improvement.

**Bootstrap confidence intervals** resample your per-case scores to estimate uncertainty without formulas:

In [ ]:
import numpy as np

def bootstrap_ci(scores, n_boot=10_000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed)
    scores = np.asarray(scores, dtype=float)
    means = rng.choice(scores, size=(n_boot, len(scores)), replace=True).mean(axis=1)
    return scores.mean(), np.quantile(means, alpha / 2), np.quantile(means, 1 - alpha / 2)

**Paired comparison**: both systems ran on the *same* cases, so compare them case by case. Most cases are answered the same way by both; what matters is the cases that changed. A paired bootstrap on per-case differences is much more sensitive than comparing two independent intervals:

In [ ]:
def paired_bootstrap(a, b, n_boot=10_000, seed=0):
    """Mean improvement of b over a, and the fraction of resamples where b is not better."""
    rng = np.random.default_rng(seed)
    d = np.asarray(b, float) - np.asarray(a, float)
    diffs = d[rng.integers(0, len(d), size=(n_boot, len(d)))].mean(axis=1)
    return d.mean(), (diffs <= 0).mean()

In [ ]:
# Is v2 really better than v1? Compare case by case.
s1 = [r["scores"]["category"] for r in res_v1]
s2 = [r["scores"]["category"] for r in res_v2]
print("v1", bootstrap_ci(s1))
print("v2", bootstrap_ci(s2))
print("paired: mean gain, fraction of resamples where v2 is not better =", paired_bootstrap(s1, s2))
fixed = [a["id"] for a, b in zip(res_v1, res_v2) if b["scores"]["category"] > a["scores"]["category"]]
broken = [a["id"] for a, b in zip(res_v1, res_v2) if b["scores"]["category"] < a["scores"]["category"]]
print("fixed:", fixed)
print("broken:", broken)

**Always look at fixes and breaks, not just the averages.** A gain of 5 points from fixing 5 cases and breaking none is strong evidence; the same gain from fixing 9 and breaking 4 is far less certain. Model outputs also vary between runs, so for important decisions run each variant two or three times.

### Hands-on: how many cases do you need?

1. Simulate a baseline that is right on 75% of cases and a candidate that fixes 5% of all cases and breaks none. For $n$ of 50, 100, 200, 400 and 800 cases, draw 100 random eval sets and run `paired_bootstrap` on each (2,000 resamples is enough here).
2. Count how often the "not better" fraction falls below 0.05. That is the chance your eval set detects a real 5-point improvement.
3. Repeat for a candidate that fixes 9% and breaks 4%, the same net gain, and print both tables.

You are done when the two tables print. Notice how many cases the noisier 5-point gain needs before you can trust it.

In [ ]:
# Your code here

## Regression testing and CI

Treat the eval suite like a test suite:

- **Fast checks on every commit**: schema validity, a small smoke set of 20 to 50 cases, safety cases. Fail the build on hard-rule violations.
- **Full evaluation before merging** prompt or model changes, with the report attached to the review.
- **Thresholds per slice**, not just overall: "unanswerable questions must stay above 95% abstention" protects behaviour that a global average would hide.
- **Pin everything**: model version, prompt version, retrieval index version, so a result can be reproduced.
- **Re-run on provider model updates** before switching.

## Public benchmarks and their limits

Benchmarks such as MMLU, GPQA and SWE-bench are useful for shortlisting models. They are not a substitute for your own evals, because of contamination, saturation, mismatch with your task, and optimisation pressure. Your eval set decides.

### Contamination in your own evals

Your own eval cases leak too: into few-shot examples, into fine-tuning data, or into a retrieval index tuned until every eval question hits. Store eval cases separately, never let a training script read the eval file, and check for near-duplicates between the two sets before every fine-tune. Lesson 19 covers decontamination in depth.

## Online evaluation

Offline sets go stale. In production, also measure explicit feedback, implicit signals (retries, rephrasing, escalations), sampled expert review, A/B tests and drift. Then feed what you learn back into the offline eval set: production to dataset to improvement to production.

### Hands-on: an eval suite for your RAG system

Using your system from lesson 14:

1. Turn your evaluation questions into `eval_cases.jsonl` with tags (`lookup`, `comparison`, `unanswerable`, and so on). Aim for at least 40 cases.
2. Implement graders: retrieval hit (gold chunk in top 5), abstention (for `unanswerable` cases, did the answer say the documents do not contain it?), and an LLM faithfulness judge.
3. Hand-grade faithfulness for 20 answers and measure your judge's agreement with you. Refine the rubric until it agrees on at least 17 of 20.
4. Compare two variants (for example, with and without the reranker) using `run_eval` and `paired_bootstrap`. List the cases fixed and broken.

In [ ]:
# Your code here

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/17-evaluation.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.